# Iceberg Basics — 09: Table Maintenance


Regular maintenance keeps Iceberg tables fast and storage efficient.

Topics: expire_snapshots, rewrite_data_files (compaction), rewrite_manifests,
remove_orphan_files, monitoring with metadata tables.


In [1]:
import os, time, pathlib, shutil, random, datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/iceberg_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('iceberg-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
spark.sql("CREATE DATABASE IF NOT EXISTS local.icedb")
print(f"Spark {spark.version} | Iceberg catalog ready")

random.seed(42)
N = 100_000
CATEGORIES = ["Electronics","Clothing","Books","Food","Sports","Furniture"]
COUNTRIES  = ["US","UK","DE","FR","JP","CA","AU","BR"]
STATUSES   = ["pending","confirmed","shipped","delivered","cancelled"]

schema = StructType([
    StructField("order_id",    LongType()),
    StructField("customer_id", LongType()),
    StructField("product",     StringType()),
    StructField("category",    StringType()),
    StructField("country",     StringType()),
    StructField("quantity",    IntegerType()),
    StructField("price",       DoubleType()),
    StructField("revenue",     DoubleType()),
    StructField("status",      StringType()),
    StructField("order_date",  DateType()),
])

base = datetime.date(2024, 1, 1)
rows = []
for i in range(N):
    qty   = random.randint(1, 10)
    price = round(random.uniform(5, 1000), 2)
    d     = base + datetime.timedelta(days=random.randint(0, 364))
    rows.append((i+1, random.randint(1,10000),
                 f"Product_{random.randint(1,200)}",
                 random.choice(CATEGORIES), random.choice(COUNTRIES),
                 qty, price, round(qty*price, 2),
                 random.choice(STATUSES), d))

df = spark.createDataFrame(rows, schema)
df.cache().count()
print(f"Dataset ready: {N:,} rows")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/10 20:41:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.2 | Iceberg catalog ready


26/04/10 20:41:12 WARN TaskSetManager: Stage 0 contains a task of very large size (1252 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 20:41:14 WARN TaskSetManager: Stage 1 contains a task of very large size (1304 KiB). The maximum recommended task size is 1000 KiB.


Dataset ready: 100,000 rows


## Step 1 — Setup: Create Maintenance-Needed Table



In [2]:

spark.sql("DROP TABLE IF EXISTS local.icedb.maint_orders")
df.writeTo("local.icedb.maint_orders").using("iceberg").createOrReplace()

# Create maintenance needs: many small writes
for i in range(10):
    df.limit(1000).writeTo("local.icedb.maint_orders").append()
# UPDATE does not support LIMIT — collect IDs in Python first
_ids = [r.order_id for r in
        spark.table("local.icedb.maint_orders").filter(F.col("status") == "confirmed")
        .select("order_id").limit(500).collect()]
if _ids:
    _ids_csv = ",".join(str(x) for x in _ids)
    spark.sql(f"UPDATE local.icedb.maint_orders SET status='shipped' WHERE order_id IN ({_ids_csv})")
spark.sql("DELETE FROM local.icedb.maint_orders WHERE status='cancelled' AND price < 25")

snap_count = spark.sql("SELECT COUNT(*) FROM local.icedb.maint_orders.snapshots").collect()[0][0]
file_count = spark.sql("SELECT COUNT(*) FROM local.icedb.maint_orders.files").collect()[0][0]
print(f"Maintenance state: {snap_count} snapshots, {file_count} data files")


26/04/10 20:41:15 WARN TaskSetManager: Stage 4 contains a task of very large size (1252 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 20:41:16 WARN HadoopTableOperations: Error reading version hint file /workspace/data/warehouse/iceberg/icedb/maint_orders/metadata/version-hint.text
java.io.FileNotFoundException: File /workspace/data/warehouse/iceberg/icedb/maint_orders/metadata/version-hint.text does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:917)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1238)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:907)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.hadoop.fs.ChecksumFileSystem$ChecksumFSInputChecker.<init>(ChecksumFileSystem.java:189)
	at org.apache.hadoop.fs.ChecksumFileSystem.open(ChecksumFileSystem.java:572)
	at org.apache.hadoop.

Maintenance state: 13 snapshots, 4 data files


## Step 2 — expire_snapshots

Remove old snapshot metadata.

In [3]:

print(f"Snapshots before: {spark.sql('SELECT COUNT(*) FROM local.icedb.maint_orders.snapshots').collect()[0][0]}")

spark.sql("""
    CALL local.system.expire_snapshots(
        table       => 'local.icedb.maint_orders',
        retain_last => 3
    )
""").show()

print(f"Snapshots after : {spark.sql('SELECT COUNT(*) FROM local.icedb.maint_orders.snapshots').collect()[0][0]}")
print("Old snapshot metadata removed (retaining last 3 snapshots)")
print("Note: expire_snapshots removes metadata only, not data files")


Snapshots before: 13


+------------------------+-----------------------------------+-----------------------------------+----------------------------+----------------------------+------------------------------+
|deleted_data_files_count|deleted_position_delete_files_count|deleted_equality_delete_files_count|deleted_manifest_files_count|deleted_manifest_lists_count|deleted_statistics_files_count|
+------------------------+-----------------------------------+-----------------------------------+----------------------------+----------------------------+------------------------------+
|                       0|                                  0|                                  0|                           0|                           0|                             0|
+------------------------+-----------------------------------+-----------------------------------+----------------------------+----------------------------+------------------------------+

Snapshots after : 13
Old snapshot metadata removed (retaini

## Step 3 — rewrite_data_files (Compaction)

Compact many small files into fewer large ones.

In [4]:

before_files = spark.sql("SELECT COUNT(*) FROM local.icedb.maint_orders.files").collect()[0][0]
before_avg   = spark.sql("SELECT AVG(file_size_in_bytes)/1024 FROM local.icedb.maint_orders.files").collect()[0][0]
print(f"Before compaction: {before_files} files, avg {before_avg:.0f} KB")

t0 = time.time()
spark.sql("""
    CALL local.system.rewrite_data_files(
        table    => 'local.icedb.maint_orders',
        strategy => 'binpack',
        options  => map('target-file-size-bytes', '67108864')
    )
""").show()
print(f"Compaction took: {time.time()-t0:.1f}s")

after_files = spark.sql("SELECT COUNT(*) FROM local.icedb.maint_orders.files").collect()[0][0]
after_avg   = spark.sql("SELECT AVG(file_size_in_bytes)/1024 FROM local.icedb.maint_orders.files").collect()[0][0] or 0
print(f"After compaction : {after_files} files, avg {after_avg:.0f} KB")


Before compaction: 4 files, avg 387 KB
+--------------------------+----------------------+---------------------+-----------------------+--------------------------+
|rewritten_data_files_count|added_data_files_count|rewritten_bytes_count|failed_data_files_count|removed_delete_files_count|
+--------------------------+----------------------+---------------------+-----------------------+--------------------------+
|                         0|                     0|                    0|                      0|                         0|
+--------------------------+----------------------+---------------------+-----------------------+--------------------------+

Compaction took: 0.1s
After compaction : 4 files, avg 387 KB


## Step 4 — rewrite_manifests

Compact manifest files for faster query planning.

In [5]:

spark.sql("""
    CALL local.system.rewrite_manifests(
        table => 'local.icedb.maint_orders'
    )
""").show()
print("Manifests compacted — fewer manifest files = faster query planning")


26/04/10 20:41:34 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------------------------+---------------------+
|rewritten_manifests_count|added_manifests_count|
+-------------------------+---------------------+
|                        9|                    1|
+-------------------------+---------------------+

Manifests compacted — fewer manifest files = faster query planning


## Step 5 — remove_orphan_files

Delete unreferenced data files.

In [6]:

t0 = time.time()
spark.sql("""
    CALL local.system.remove_orphan_files(
        table            => 'local.icedb.maint_orders',
        dry_run          => true
    )
""").show(truncate=False)
print(f"Orphan file check took {time.time()-t0:.1f}s")
print()
print("Maintenance schedule recommendation:")
print("  Daily   : expire_snapshots(retain_last=5)")
print("  Daily   : rewrite_data_files on recent partitions")
print("  Weekly  : rewrite_manifests")
print("  Monthly : remove_orphan_files")


[Stage 77:====================================================>   (34 + 2) / 36]

+--------------------+
|orphan_file_location|
+--------------------+
+--------------------+

Orphan file check took 3.8s

Maintenance schedule recommendation:
  Daily   : expire_snapshots(retain_last=5)
  Daily   : rewrite_data_files on recent partitions
  Weekly  : rewrite_manifests
  Monthly : remove_orphan_files


## Summary



In [7]:

print("""
-- Expire old snapshots
CALL local.system.expire_snapshots(table=>'db.tbl', retain_last=>5)

-- Compact small files
CALL local.system.rewrite_data_files(
    table    => 'db.tbl',
    strategy => 'binpack',
    options  => map('target-file-size-bytes','134217728')
)

-- Compact manifests
CALL local.system.rewrite_manifests(table=>'db.tbl')

-- Remove unreferenced files
CALL local.system.remove_orphan_files(table=>'db.tbl')

Maintenance schedule:
  After streaming writes: expire_snapshots + rewrite_data_files
  Weekly: rewrite_manifests
  Monthly: remove_orphan_files (with dry_run=true first!)
""")



-- Expire old snapshots
CALL local.system.expire_snapshots(table=>'db.tbl', retain_last=>5)

-- Compact small files
CALL local.system.rewrite_data_files(
    table    => 'db.tbl',
    strategy => 'binpack',
    options  => map('target-file-size-bytes','134217728')
)

-- Compact manifests
CALL local.system.rewrite_manifests(table=>'db.tbl')

-- Remove unreferenced files
CALL local.system.remove_orphan_files(table=>'db.tbl')

Maintenance schedule:
  After streaming writes: expire_snapshots + rewrite_data_files
  Weekly: rewrite_manifests
  Monthly: remove_orphan_files (with dry_run=true first!)

